# **CLEANING CSV: pizza_restaurant_dataset_raw**

## **Load + View Dataset**

In [82]:
# Import libraries
import pandas as pd
import numpy as np
import re

# Load dataset
df = pd.read_csv("pizza_restaurant_dataset_raw.csv")

In [83]:
# Inspect dataset
print("Dataset Dimensions: ", df.shape)
print("Dataset Attributes: ", df.columns.values)

df.info()
df.head(10)

Dataset Dimensions:  (5050, 11)
Dataset Attributes:  ['Order ID' 'Order Date' 'Menu Item Name' 'Menu Item Category'
 'Menu Item Price' 'Menu Item Size' 'Menu Item Quantity'
 'Customer First Name' 'Customer Last Name' 'Delivery' 'Delivery Address']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5050 entries, 0 to 5049
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Order ID             5050 non-null   object 
 1   Order Date           5050 non-null   object 
 2   Menu Item Name       5050 non-null   object 
 3   Menu Item Category   5050 non-null   object 
 4   Menu Item Price      5050 non-null   float64
 5   Menu Item Size       5010 non-null   object 
 6   Menu Item Quantity   5050 non-null   int64  
 7   Customer First Name  5050 non-null   object 
 8   Customer Last Name   5010 non-null   object 
 9   Delivery             5050 non-null   object 
 10  Delivery Address     3245 non-null   obj

,Order ID,Order Date,Menu Item Name,Menu Item Category,Menu Item Price,Menu Item Size,Menu Item Quantity,Customer First Name,Customer Last Name,Delivery,Delivery Address
0,QK51FPKH1D,2024-01-13,Crazy Bread-,Sides,4.99,Regular,6,Nancy,Moore,Yes,"1779 Eastern Pkwy, Brooklyn, NY"
1,5UZBIKCIDK,2024-07-02,ExtraMostBestest Cheese,Pizza,10.49,Large,3,Nancy,Anderson,No,NaN
2,41IBLJH75L,2024-07-05,Crazy Bread,Sides,4.99,Regular,2,Ashley,Williams,Yes,"4474 Utica Ave, Brooklyn, NY"
3,4FFYVNVQJT,2024-07-21,Crazy Bread,Sides,4.99,Regular,7,William,Jackson,Yes,"9297 Court St, Brooklyn, NY"
4,4SHNF877VR,2024-12-14,Mountain Dew 2-Liter,Beverage,3.29,2L,7,DAVID,Brown,yes,NaN
5,871YZOYN6Q,2024-07-04,Classic Pepperoni Pizza,Pizza,9.49,Large,3,John,Thomas,Yes,"9395 Atlantic Ave, Brooklyn, NY"
6,UR23GDPPQ0,2024-02-02,ExtraMostBestest Pepperoni,Pizza,10.49,Large,3,Linda,Smith,Yes,"3870 Ocean Pkwy, Brooklyn, NY"
7,5TB94874FR,2024-11-26,Crazy Bread Combo,Sides,5.99,Regular,1,Linda,Miller,Yes,"5659 Atlantic Ave, Brooklyn, NY"
8,HC0CLBR7TC,2024-07-24,Classic Pepperoni Pizza,Pizza,9.49,Large,1,Ashley,Anderson,Yes,"6496 Church Ave, Brooklyn, NY"
9,HZIOYKL1CQ,2024-04-05,Deep Dish Pepperoni,Pizza,11.99,Regular,1,Patricia,Jackson,Yes,"3953 Coney Island Ave, Brooklyn, NY"


## **ISSUE #1: Fix Column Names + Data Types**
### Convert to snake_case to improve consistency and readability:
* Lowercase
* Remove leading/trailing whitespace
* Remove parentheses
* Replace dashes/spaces with underscores

In [84]:
# Standardize column names
df.columns = (df.columns
    .str.lower()
    .str.strip()
    .str.replace(" ", "_")
    .str.replace("-", "_")
    .str.replace("(", "")
    .str.replace(")", "")
)

# Inspect columns
df.columns

Index(['order_id', 'order_date', 'menu_item_name', 'menu_item_category',
       'menu_item_price', 'menu_item_size', 'menu_item_quantity',
       'customer_first_name', 'customer_last_name', 'delivery',
       'delivery_address'],
      dtype='object')

## **ISSUE #2: Fix Duplicate Values**
### Remove duplicate values to ensure accurate data.

In [85]:
# Count duplicate values
print("Duplicates Count: ", df.duplicated().sum())

# No duplicates so dropping duplicate rows is unnecessary

Duplicates Count:  0


## **ISSUE #3: Fix `menu_item_category`, `menu_item_size`, and `delivery` Values**
### Some values in the **`menu_item_category`**, **`menu_item_size`**, and **`delivery`** columns have typos, or are recorded in an inconsistent manner.

In [86]:
# Find unique values in menu_item_category, menu_item_size, and delivery columns to pinpoint inconsistencies

print("Unique Category Values: ", df["menu_item_category"].unique())
print("Unique Size Values: ", df["menu_item_size"].unique())
print("Unique Delivery Values: ", df["delivery"].unique())

Unique Category Values:  ['Sides' 'Pizza' 'Beverage' 'Wings' 'Bev' 'Sidse' 'Wing' 'Piza' 'pzza']
Unique Size Values:  ['Regular' 'Large' '2L' '20 oz' '8 pc' nan 'reg' '8pc' 'Lrage' '2l' 'smll']
Unique Delivery Values:  ['Yes' 'No' 'yes' 'Y' 'no' 'N']


### The values within these columns will be corrected and simplified, in order to ensure accurate data when querying and making visualizations:
* **`menu_item_category`** → Convert all records to either **`Pizza`**, **`Sides`**, **`Wings`**, or **`Beverage`**
* **`menu_item_size`** → Convert all records to either **`Regular`**, **`Large`**, **`Small`**, **`2L`**, **`20oz`**, or **`8pc`**
* **`delivery`** → Convert all records to either **`Yes`** or **`No`**

In [87]:
# Correct and simplify values in menu_item_category column
df["menu_item_category"] = df["menu_item_category"].replace(["Piza", "pzza"], "Pizza")
df["menu_item_category"] = df["menu_item_category"].replace("Sidse", "Sides")
df["menu_item_category"] = df["menu_item_category"].replace("Wing", "Wings")
df["menu_item_category"] = df["menu_item_category"].replace("Bev", "Beverage")

# Correct and simplify values in menu_item_size column
df["menu_item_size"] = df["menu_item_size"].replace("reg", "Regular")
df["menu_item_size"] = df["menu_item_size"].replace("Lrage", "Large")
df["menu_item_size"] = df["menu_item_size"].replace("smll", "Small")
df["menu_item_size"] = df["menu_item_size"].replace("2l", "2L")
df["menu_item_size"] = df["menu_item_size"].replace("20 oz", "20oz")
df["menu_item_size"] = df["menu_item_size"].replace("8 pc", "8pc")

# Correct and simplify values in delivery column
df["delivery"] = df["delivery"].replace(["yes", "Y"], "Yes")
df["delivery"] = df["delivery"].replace(["no", "N"], "No")

# Make sure values in menu_item_category, menu_item_size, and delivery columns are now consistent
print("Unique Category Values: ", df["menu_item_category"].unique())
print("Unique Size Values: ", df["menu_item_size"].unique())
print("Unique Delivery Values: ", df["delivery"].unique())

Unique Category Values:  ['Sides' 'Pizza' 'Beverage' 'Wings']
Unique Size Values:  ['Regular' 'Large' '2L' '20oz' '8pc' nan 'Small']
Unique Delivery Values:  ['Yes' 'No']


## **ISSUE #4: Fix `menu_item_name`, `customer_first_name`, `customer_last_name`, and `delivery_address` Values**
### Some values in the **`menu_item_name`**, **`customer_first_name`**, **`customer_last_name`**, and **`delivery_address`** columns have inconsistent capitalization, or random symbols that don't belong. This will be corrected, while preserving the descriptive text, in order to prevent parsing errors.

In [88]:
# Identify the columns that require cleaning
cols_to_clean = [
    "menu_item_name",
    "customer_first_name",
    "customer_last_name",
    "delivery_address"
]

# Identify the symbols that must be removed
unwanted_symbols = r'[\/:;\(\)\-\"“”‘’\'\\%]'

# Define a function that cleans the columns
def clean_text(val):

    if pd.isna(val):
        return val
    
    # Remove unwanted symbols
    val = re.sub(unwanted_symbols, '', str(val))
    
    # Normalize extra spaces
    val = re.sub(r'\s+', ' ', val).strip()
    
    # Convert words to title case
    return val.title()

# Apply the function to the columns
for col in cols_to_clean:
    df[col] = df[col].apply(clean_text)

# Make sure text columns were cleaned
print(df[cols_to_clean].head())

            menu_item_name customer_first_name customer_last_name  \
0              Crazy Bread               Nancy              Moore   
1  Extramostbestest Cheese               Nancy           Anderson   
2              Crazy Bread              Ashley           Williams   
3              Crazy Bread             William            Jackson   
4      Mountain Dew 2Liter               David              Brown   

                  delivery_address  
0  1779 Eastern Pkwy, Brooklyn, Ny  
1                              NaN  
2     4474 Utica Ave, Brooklyn, Ny  
3      9297 Court St, Brooklyn, Ny  
4                              NaN  


In [89]:
# Fix strings that were undesirably altered during the cleaning process
df["menu_item_name"] = df["menu_item_name"].replace("2Liter", "2 Liter", regex=True)
df["menu_item_name"] = df["menu_item_name"].replace("Extramostbestest", "ExtraMostBestest", regex=True)
df["delivery_address"] = df["delivery_address"].replace("Ny", "NY", regex=True)

# Make sure text columns were cleaned
print("Unique Item Values: ", df["menu_item_name"].unique())
print(df["delivery_address"].head(10))

Unique Item Values:  ['Crazy Bread' 'ExtraMostBestest Cheese' 'Mountain Dew 2 Liter'
 'Classic Pepperoni Pizza' 'ExtraMostBestest Pepperoni'
 'Crazy Bread Combo' 'Deep Dish Pepperoni' 'Aquafina Water'
 '2 Liter Diet Pepsi' 'Buffalo Wings' 'Caesar Wings' 'Supreme Pizza'
 'Veggie Pizza' 'Classic Cheese Pizza' 'Deep Dish Cheese' '2 Liter Pepsi'
 'Italian Cheese Bread' '3 Meat Treat Pizza']
0        1779 Eastern Pkwy, Brooklyn, NY
1                                    NaN
2           4474 Utica Ave, Brooklyn, NY
3            9297 Court St, Brooklyn, NY
4                                    NaN
5        9395 Atlantic Ave, Brooklyn, NY
6          3870 Ocean Pkwy, Brooklyn, NY
7        5659 Atlantic Ave, Brooklyn, NY
8          6496 Church Ave, Brooklyn, NY
9    3953 Coney Island Ave, Brooklyn, NY
Name: delivery_address, dtype: object


## **ISSUE #5: Create `customer_name` Column**
### In order to simplify the dataset, combine each value in the **`customer_first_name`** and **`customer_last_name`** columns, to create one **`customer_name`** column that contains the customer's full name.

In [90]:
# Combine customer_first_name and customer_last_name columns into customer_name
df["customer_name"] = (
    df["customer_first_name"].fillna("") + " " + df["customer_last_name"].fillna("")
).str.strip()

# Place new customer_name column into the correct position
insert_pos = df.columns.get_loc("menu_item_quantity") + 1
col_data = df.pop("customer_name")
df.insert(insert_pos, "customer_name", col_data)

# Drop unnecessary columns
df = df.drop(columns=["customer_first_name", "customer_last_name"])

# Inspect customer_name column
print(df["customer_name"].head(10))

0         Nancy Moore
1      Nancy Anderson
2     Ashley Williams
3     William Jackson
4         David Brown
5         John Thomas
6         Linda Smith
7        Linda Miller
8     Ashley Anderson
9    Patricia Jackson
Name: customer_name, dtype: object


## **View Cleaned Dataset**

In [91]:
# Replace any remaining empty fields with "", to prevent errors during parsing
df = df.replace(np.nan, "")

# Inspect dataset
print("Dataset Dimensions: ", df.shape)
print("Dataset Attributes: ", df.columns.values)

df.info()
df.head(10)

Dataset Dimensions:  (5050, 10)
Dataset Attributes:  ['order_id' 'order_date' 'menu_item_name' 'menu_item_category'
 'menu_item_price' 'menu_item_size' 'menu_item_quantity' 'customer_name'
 'delivery' 'delivery_address']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5050 entries, 0 to 5049
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   order_id            5050 non-null   object 
 1   order_date          5050 non-null   object 
 2   menu_item_name      5050 non-null   object 
 3   menu_item_category  5050 non-null   object 
 4   menu_item_price     5050 non-null   float64
 5   menu_item_size      5050 non-null   object 
 6   menu_item_quantity  5050 non-null   int64  
 7   customer_name       5050 non-null   object 
 8   delivery            5050 non-null   object 
 9   delivery_address    5050 non-null   object 
dtypes: float64(1), int64(1), object(8)
memory usage: 394.7+ KB


,order_id,order_date,menu_item_name,menu_item_category,menu_item_price,menu_item_size,menu_item_quantity,customer_name,delivery,delivery_address
0,QK51FPKH1D,2024-01-13,Crazy Bread,Sides,4.99,Regular,6,Nancy Moore,Yes,"1779 Eastern Pkwy, Brooklyn, NY"
1,5UZBIKCIDK,2024-07-02,ExtraMostBestest Cheese,Pizza,10.49,Large,3,Nancy Anderson,No,
2,41IBLJH75L,2024-07-05,Crazy Bread,Sides,4.99,Regular,2,Ashley Williams,Yes,"4474 Utica Ave, Brooklyn, NY"
3,4FFYVNVQJT,2024-07-21,Crazy Bread,Sides,4.99,Regular,7,William Jackson,Yes,"9297 Court St, Brooklyn, NY"
4,4SHNF877VR,2024-12-14,Mountain Dew 2 Liter,Beverage,3.29,2L,7,David Brown,Yes,
5,871YZOYN6Q,2024-07-04,Classic Pepperoni Pizza,Pizza,9.49,Large,3,John Thomas,Yes,"9395 Atlantic Ave, Brooklyn, NY"
6,UR23GDPPQ0,2024-02-02,ExtraMostBestest Pepperoni,Pizza,10.49,Large,3,Linda Smith,Yes,"3870 Ocean Pkwy, Brooklyn, NY"
7,5TB94874FR,2024-11-26,Crazy Bread Combo,Sides,5.99,Regular,1,Linda Miller,Yes,"5659 Atlantic Ave, Brooklyn, NY"
8,HC0CLBR7TC,2024-07-24,Classic Pepperoni Pizza,Pizza,9.49,Large,1,Ashley Anderson,Yes,"6496 Church Ave, Brooklyn, NY"
9,HZIOYKL1CQ,2024-04-05,Deep Dish Pepperoni,Pizza,11.99,Regular,1,Patricia Jackson,Yes,"3953 Coney Island Ave, Brooklyn, NY"


## **Save Cleaned Dataset**

In [92]:
# Save cleaned dataset
df.to_csv("pizza_restaurant_dataset_cleaned.csv", index=False)